In [2]:
# Install the required libraries
!pip install transformers

# Import necessary libraries
from transformers import BioBertModel, BertTokenizer
import torch

# Initialize BioBERT (this will download ~400MB model)
model = BioBertModel.from_pretrained("monologg/biobert_v1.1_pubmed")  # Medical-domain BERT
tokenizer = BertTokenizer.from_pretrained("monologg/biobert_v1.1_pubmed")

# Test GPU availability
print("GPU available:", torch.cuda.is_available())  # Should return True


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

GPU available: True


In [4]:
!nvidia-smi


Sun May 25 13:50:58 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# ✅ Step 1: Import libraries (use torch.cuda.amp here for compatibility with Colab)
from torch.cuda.amp import GradScaler, autocast
from transformers import BertModel, BertTokenizer
import torch
from torch.optim import AdamW

# ✅ Step 2: Load the BioBERT model and tokenizer
model = BertModel.from_pretrained("monologg/biobert_v1.1_pubmed")
tokenizer = BertTokenizer.from_pretrained("monologg/biobert_v1.1_pubmed")

# ✅ Step 3: Prepare a dummy input sentence
input_text = "COVID-19 is caused by the SARS-CoV-2 virus."
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

# ✅ Step 4: Move model and inputs to GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

# ✅ Step 5: Initialize optimizer
optimizer = AdamW(model.parameters(), lr=1e-5)

# ✅ Step 6: Initialize GradScaler (older way for compatibility)
scaler = GradScaler()  # No arguments for older torch versions (like in Colab)

# ✅ Step 7: Simulated training step using AMP
model.train()
with autocast():  # Also from torch.cuda.amp (older syntax)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    # Fake loss: mean of last hidden states (just for testing AMP)
    loss = outputs.last_hidden_state.mean()

# ✅ Step 8: Simulate backward + optimizer step
scaler.scale(loss).backward()   # Compute gradients with scaled loss
scaler.step(optimizer)          # Update model weights
scaler.update()                 # Update scaler for next iteration


<ipython-input-8-4407b5b6e69c>:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # No arguments for older torch versions (like in Colab)
<ipython-input-8-4407b5b6e69c>:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # Also from torch.cuda.amp (older syntax)


In [9]:
# ✅ Step 8: Save the trained (or initialized) model and tokenizer to a directory
# This allows future loading without re-downloading everything.

# ✅ Choose a save path — you can change this later to suit your project
save_path = "data/models/biobert-base"

# ✅ Save model weights and configuration to the given path
model.save_pretrained(save_path)

# ✅ Save tokenizer config (vocab, tokenizer type, etc.) to same path
tokenizer.save_pretrained(save_path)

print(f"✅ Model and tokenizer saved to: {save_path}")


✅ Model and tokenizer saved to: data/models/biobert-base


In [2]:
from transformers import AutoTokenizer, AutoModel

model_path = "../../data/models/biobert-base"  # go up two folders to reach /data/

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)

print("✅ BioBERT model and tokenizer loaded locally.")


✅ BioBERT model and tokenizer loaded locally.


In [3]:
text = "The patient was prescribed amoxicillin and developed a rash."

inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

print("✅ Inference done.")
print("Hidden states shape:", outputs.last_hidden_state.shape)


✅ Inference done.
Hidden states shape: torch.Size([1, 16, 768])
